<h3 style="color:blue;">Importation des bibliothèques</h3> 

Dans cette étape, nous importons les bibliothèques nécessaires pour la phase de vectorisation du texte.

- **sqlite3 :** permet de se connecter et d’extraire les données depuis la base de données SQLite contenant les documents textuels.

- **joblib :** utilisé pour sauvegarder le modèle TF-IDF une fois entraîné, afin de le réutiliser dans d’autres parties du projet (classification ou API).

- **TfidfVectorizer :** outil de scikit-learn permettant de transformer les textes en une représentation numérique basée sur la fréquence des mots pondérée par leur importance (TF-IDF).

Cette étape prépare l’environnement pour la transformation des données textuelles en vecteurs exploitables par les modèles de machine learning.

In [4]:
import sqlite3
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

<h3 style="color:blue;">Paramétrage global</h3>

Dans cette étape, nous définissons les paramètres essentiels utilisés dans la phase de lecture et de traitement des données.

- **DB_PATH :** définit le chemin vers la base de données SQLite contenant les documents textuels. Cela permet de centraliser l’accès aux données et de faciliter la réutilisation du code.

- **BATCH_SIZE :** définit le nombre de lignes (documents) traitées à chaque itération lors du chargement des données. Ici, les données sont traitées par blocs de 1000 lignes afin d’éviter la surcharge mémoire et d’optimiser les performances.

Cette approche par batch permet de gérer efficacement de grandes bases de données sans charger l’ensemble des données en mémoire simultanément.

In [5]:
DB_PATH = "../Data/dataset.db"
BATCH_SIZE = 1000

<h3 style="color:blue;">Chargement des données par lots depuis SQLite</h3> 

Cette fonction lit les textes depuis la base de données de manière progressive.

Elle utilise fetchmany (BATCH_SIZE) pour récupérer les données par blocs de 1000 lignes afin d’éviter de charger toute la base en mémoire.

Le mot-clé yield permet de retourner les données par lots, ce qui rend la fonction plus efficace pour le traitement de grandes bases de données.

Enfin, la connexion à la base est fermée à la fin du traitement.

In [6]:
def fetch_batches():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT clean_text FROM documents")

    while True:
        batch = cursor.fetchmany(BATCH_SIZE)
        if not batch:
            break
        yield batch

    conn.close()

<h3 style="color:blue;">Constitution d’un échantillon pour l’apprentissage TF-IDF</h3> 

Cette étape consiste à créer un échantillon représentatif des données textuelles afin d’entraîner le vectoriseur TF-IDF.

Les données sont parcourues par lots grâce à la fonction **fetch_batches()**.

Chaque texte non vide est ajouté à une liste temporaire **sample_texts**.

Afin d’éviter une surcharge mémoire, le traitement est limité à **100 batches**.

Cet échantillon est ensuite utilisé pour construire le vocabulaire TF-IDF de manière efficace et contrôlée.

In [7]:
print("Collecte d'un échantillon pour TF-IDF...")

sample_texts = []

for i, batch in enumerate(fetch_batches()):
    for (text,) in batch:
        if text:
            sample_texts.append(text)

    # limite volontaire pour éviter RAM overflow
    if i == 100:  
        break

print("Échantillon taille:", len(sample_texts))

Collecte d'un échantillon pour TF-IDF...
Échantillon taille: 101000


<h3 style="color:blue;">Construction du modèle TF-IDF</h3> 

Cette étape consiste à transformer les textes en une représentation numérique basée sur la méthode TF-IDF.

Le vectoriseur est configuré avec :

- **max_features=5000 :** limite le vocabulaire aux 5000 mots les plus pertinents du corpus.

- **min_df=5 :** ignore les mots trop rares apparaissant dans très peu de documents.

- **max_df=0.8 :** ignore les mots trop fréquents qui n’apportent pas d’information discriminante.

Le modèle est ensuite entraîné sur l’échantillon de textes afin de construire le vocabulaire et préparer la transformation des documents en vecteurs numériques.

Cette étape permet de représenter le texte sous forme exploitable par les algorithmes de machine learning.

In [8]:
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8
)

tfidf.fit(sample_texts)

print("TF-IDF entraîné (vocabulaire construit)")

TF-IDF entraîné (vocabulaire construit)


<h3 style="color:blue;">Vectorisation des documents par batch</h3> 

Cette étape consiste à transformer les textes en représentations numériques à l’aide du modèle TF-IDF déjà entraîné.

Les données sont traitées par lots afin d’éviter une surcharge mémoire :

- Chaque batch est récupéré depuis la base de données via fetch_batches().
- Les textes vides sont remplacés par des chaînes vides pour éviter les erreurs.
- Chaque lot de textes est transformé en vecteurs TF-IDF grâce à la méthode transform().
- Les matrices obtenues sont stockées progressivement dans une liste.

Cette approche permet de vectoriser un grand volume de données de manière efficace sans charger l’ensemble du dataset en mémoire.

In [9]:
import numpy as np
from scipy.sparse import vstack

X_batches = []

print("Vectorisation par batch...")

for batch in fetch_batches():

    texts = [text if text else "" for (text,) in batch]

    X = tfidf.transform(texts)

    X_batches.append(X)

print("Vectorisation terminée")

Vectorisation par batch...
Vectorisation terminée


<h3 style="color:blue;">Fusion des matrices de caractéristiques</h3> 

Cette étape consiste à regrouper tous les vecteurs TF-IDF générés par batch en une seule matrice globale.

- Les matrices partielles stockées dans **X_batches** sont combinées à l’aide de **vstack**.
- Cette opération permet de reconstruire une matrice unique représentant l’ensemble des documents.
- Le résultat final est une matrice creuse (sparse matrix) adaptée au traitement de grands volumes de données.

La forme finale de la matrice est ensuite affichée pour vérifier la dimension des données vectorisées.

In [10]:
print("Fusion des matrices...")

X = vstack(X_batches)

print("Shape finale:", X.shape)

Fusion des matrices...
Shape finale: (125248, 5000)


<h3 style="color:blue;">Sauvegarde du modèle TF-IDF</h3> 

Cette étape permet de sauvegarder le modèle TF-IDF entraîné afin de le réutiliser ultérieurement sans avoir à le recalculer.

- Le vectoriseur est enregistré dans un fichier au format .pkl à l’aide de joblib.
- Cela permet de garantir la cohérence entre la phase d’entraînement et les futures prédictions.
- Le modèle sauvegardé pourra être utilisé dans d’autres parties du projet.

Cette opération est essentielle pour la réutilisation et le déploiement du système.

In [11]:
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Vectorizer sauvegardé")

Vectorizer sauvegardé


<h3 style="color:blue;">Visualisation d’une partie de la matrice TF-IDF</h3> 

Cette étape permet d’afficher une portion réduite de la matrice TF-IDF afin de rendre les résultats plus lisibles et compréhensibles.

In [13]:
import pandas as pd

feature_names = tfidf.get_feature_names_out()

X_small = X[:10, 2350:3750].toarray()

df_view = pd.DataFrame(
    X_small,
    columns=feature_names[2350:3750]
)

# garder uniquement les colonnes avec valeurs non nulles
df_view = df_view.loc[:, (df_view != 0).any(axis=0)]

df_view

,interest,introduced,introduction,islamic,issue,italy,iv,jm,jun,kg,...,program,property,proposed,public,pupil,questionnaire,race,rapidly,rat,reached
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.081759,0.064974,0.088485,...,0.000000,0.076226,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.169968,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
3,0.000000,0.089377,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.395,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
5,0.066393,0.000000,0.046721,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.067379,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,0.107492,0.073292,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.064497,0.000000,0.000000,0.000000,0.118233,0.000000,0.000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.065071,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.038279
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.27326,0.08372,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.072031,0.000000,0.000000,0.148247,0.000,0.078325,0.000000,0.000000
